# Finding the Fastest Route Across Central Ho Chi Minh City

It answers a simple question - "what's the fastest way from A to B right now?" - for real places in central Ho Chi Minh City (District 1, 3, 4, 5, 10, Binh Thanh, Phu Nhuan). The twist is that "fastest" depends on *when* you leave: a road that's quick at 2 AM can be a crawl at 5 PM. So instead of a fixed map of road speeds, the search uses a traffic model that changes by time of day, calibrated against live TomTom traffic data where possible.

The map itself has exactly 150 real points of interest - hospitals, schools, markets, temples, hotels, museums, parks, and more - pulled from OpenStreetMap.

In [1]:
# Bring in the project's own modules, plus the widgets/display helpers used
# for the interactive "Find Path" section further down.
from src import astar, data_collection, heuristic, traffic_model, visualization

import ipywidgets as widgets
from IPython.display import display

In [2]:
# Load the cached road network and the 150-POI dataset (both already
# fetched and committed to data/, so this runs fully offline).
graph = data_collection.load_or_download_graph()
pois = data_collection.load_or_download_pois()

# Each POI needs its nearest graph node so A* has somewhere to start/end.
if "node_id" not in pois.columns:
    pois = data_collection.snap_pois_to_graph(pois, graph)

# v_free_bound: fastest road class present in the graph, used as the
# heuristic's speed ceiling. corridor_bias: real TomTom-calibrated
# congestion multipliers per street, if calibration data exists.
v_free_bound = heuristic.compute_free_flow_bound_kmh(graph)
corridor_bias = traffic_model.load_corridor_bias()

print(f"{len(pois)} POIs loaded, v_free_bound={v_free_bound} km/h, "
      f"{len(corridor_bias)} calibrated corridors")

150 POIs loaded, v_free_bound=70.0 km/h, 12 calibrated corridors


## The heuristic: how A* guesses the remaining distance

A* needs a heuristic `h(n, t)`: a quick estimate of "how much travel time is left to reach the goal, starting from node `n` at time `t`". For A* to guarantee it finds the actual shortest path, that guess has to follow two rules:

- **Admissible** - it must never *overestimate*. Guessing "20 minutes left" when the real answer is 15 is fine (the guess is just a bit cautious). Guessing "20" when the real answer is 25 breaks the algorithm.
- **Consistent** - a bonus property: the guess shouldn't drop by more than the real cost of the step you just took. Not required for correctness, but without it A* has to double-check some nodes it thought it was already done with.

### The formula

```
h(n, t) = straight-line distance(n, goal) / (best possible speed anywhere in [t, t + 3h])
```

Take the straight-line distance to the goal (always shorter than or equal to the real road distance), and divide by the fastest speed achievable *anywhere in the next 3 hours*. That gives a lower bound on travel time: a guess that can only ever be too optimistic, never too pessimistic - which is exactly what admissibility requires.

### Why "anywhere in the next 3 hours" instead of just "right now"?

Our first version only looked at traffic *at the current moment*. That turned out to be a real bug: a road segment reached later in the trip can catch a moment when traffic has eased up, going faster than it was when the trip started. Using only the current traffic level made the heuristic *overestimate* the true cost - a genuine admissibility violation, not just a theoretical worry.

We caught this with real numbers: three points on a straight line, 2.2 km then 8.9 km apart, all on an 80 km/h road. Leaving the first point at 9:00 AM, the actual trip takes about 14.9 minutes - but the old heuristic guessed 15.2 minutes. Overestimating by even a few seconds is enough to break the guarantee.

The fix: instead of "traffic right now", use "the worst traffic you could hit at any point in the next 3 hours". Trips in this area should never realistically take anywhere near that long, so the true trip is essentially guaranteed to fit inside the window - which keeps the guess honest. (As a safety net, `astar.py` logs a warning if a computed route ever *does* take longer than 3 hours, since that specific result would fall outside this assumption.)

### So - is it admissible?

Yes, given that 3-hour assumption holds. At every point during the trip, the real speed can never beat "fastest road type in the whole map x worst-case traffic in the next 3 hours" - both are best-case numbers by construction. So the true remaining time can never be *less* than the heuristic's guess. This is checked directly in code too: `test_windowed_heuristic_is_admissible_everywhere_old_one_was_not` sweeps a full day and confirms the new heuristic holds everywhere the old one didn't.

### Is it consistent?

Almost everywhere, with two narrow exceptions. Consistency needs the traffic estimate to never *increase* as you move one step further along the route. That mostly holds, since "worst case in the next 3 hours" only shifts gradually - but right as that 3-hour window slides onto a period where traffic is about to ease up (twice a day, roughly 6:40-7:00 AM and 4:37-7:00 PM), the estimate can tick upward for a single step. That's a real, narrow inconsistency, about 4-6 minutes wide each time.

This doesn't break anything, though. Classic A* theory says admissibility alone is enough to guarantee the shortest path is found - consistency is just a bonus that normally lets A* skip re-checking nodes it already visited. Since we don't have that bonus everywhere, `astar.py` simply re-checks ("reopens") a node whenever it finds a cheaper way to reach it, even one it already processed once. That keeps every answer correct, at the small cost of occasionally re-checking a node during those two narrow windows.

In [3]:
import subprocess
import sys

# Run the heuristic test suite as evidence for the admissibility/consistency
# claims above. Uses sys.executable (this kernel's own Python) rather than a
# bare "pytest", since a bare "pytest" may not be on PATH.
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_heuristic.py", "-v"],
    capture_output=True, text=True, cwd=".",
)

# Print stdout, and stderr/return code too if something went wrong, so a
# failure is visible here instead of leaving this cell blank.
print(result.stdout[-3000:])

if result.stderr:
    print("--- stderr ---")
    print(result.stderr[-3000:])

print(f"--- pytest exited with return code {result.returncode} ---")

============================= test session starts =============================
platform win32 -- Python 3.12.10, pytest-9.0.2, pluggy-1.6.0 -- C:\Users\ASUS\AppData\Local\Programs\Python\Python312\python.exe
cachedir: .pytest_cache
metadata: {'Python': '3.12.10', 'Platform': 'Windows-11-10.0.26200-SP0', 'Packages': {'pytest': '9.0.2', 'pluggy': '1.6.0'}, 'Plugins': {'anyio': '4.14.1', 'asyncio': '1.4.0', 'cov': '7.1.0', 'html': '4.2.0', 'metadata': '3.1.1', 'timeout': '2.4.0', 'typeguard': '4.5.2'}, 'JAVA_HOME': 'C:\\Program Files\\Java\\jdk-23'}
rootdir: C:\Users\ASUS\Documents\261 Semester\Introduction to Artificial Intelligence\Assignment 1
configfile: pyproject.toml
plugins: anyio-4.14.1, asyncio-1.4.0, cov-7.1.0, html-4.2.0, metadata-3.1.1, timeout-2.4.0, typeguard-4.5.2
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collecting ... collected 7 items

tests/test_heuristic.py::test_collinear_points_are_set_u

In [4]:
# Dropdown lists of every POI name, a departure-time slider, and the
# button that triggers the search.
poi_names = sorted(pois["name"].dropna().unique().tolist())
start_box = widgets.Combobox(options=poi_names, description="Start:", ensure_option=True)
end_box = widgets.Combobox(options=poi_names, description="End:", ensure_option=True)
hour_slider = widgets.FloatSlider(value=8.0, min=0.0, max=23.9, step=0.25, description="Hour:")
run_button = widgets.Button(description="Find Path")
output = widgets.Output()


def on_click(_):
    with output:
        output.clear_output()

        # Look up the chosen POIs by name; bail out with a friendly message
        # if either box doesn't match a real POI.
        try:
            start_row = pois[pois["name"] == start_box.value].iloc[0]
            end_row = pois[pois["name"] == end_box.value].iloc[0]
        except IndexError:
            print("POI name not found: check spelling against the dropdown list.")
            return

        # Run the actual time-dependent A* search, then render and show the map.
        result = astar.find_path(graph, start_row["node_id"], end_row["node_id"],
                                  hour_slider.value, v_free_bound, corridor_bias)
        fmap = visualization.render_map(graph, pois, result, departure_hour=hour_slider.value)
        display(fmap)

        print(f"Distance: {result.total_distance_km:.2f} km, "
              f"Time: {result.total_travel_time_hours * 60:.1f} min")


run_button.on_click(on_click)
display(start_box, end_box, hour_slider, run_button, output)

Combobox(value='', description='Start:', ensure_option=True, options=('A&EM', 'AEON Tạ Quang Bửu', 'Bach hoa X…

Combobox(value='', description='End:', ensure_option=True, options=('A&EM', 'AEON Tạ Quang Bửu', 'Bach hoa XAN…

FloatSlider(value=8.0, description='Hour:', max=23.9, step=0.25)

Button(description='Find Path', style=ButtonStyle())

Output()

## Worked examples

The "Find Path" button above needs a real click to run, and an automated, non-interactive run of this notebook can't simulate that. So the three examples below run directly in code instead, guaranteeing real maps show up every time this notebook is re-executed. All three names are drawn straight from the actual 150-POI dataset:

1. **Nhà hát Thành phố** (Saigon Opera House, District 1) -> **Bảo tàng Thành phố Hồ Chí Minh** (HCMC Museum, District 1)
2. **Hồ Con Rùa** (Turtle Lake, District 3) -> **Khách sạn Caravelle Saigon** (Caravelle Hotel, District 1)
3. **Bệnh viện Từ Dũ** (Tu Du Hospital, District 1) -> **Nhà thờ Thánh Jeanne d'Arc** (Jeanne d'Arc Church, District 3)

All three depart at 8:00 AM (rush hour).

In [5]:
# Three real start/goal pairs, run directly (see the markdown above for why).
demo_pairs = [
    ("Nhà hát Thành phố", "Bảo tàng Thành phố Hồ Chí Minh"),
    ("Hồ Con Rùa", "Khách sạn Caravelle Saigon"),
    ("Bệnh viện Từ Dũ", "Nhà thờ Thánh Jeanne d\'Arc"),
]

for start_name, end_name in demo_pairs:
    start_row = pois[pois["name"] == start_name].iloc[0]
    end_row = pois[pois["name"] == end_name].iloc[0]

    demo_result = astar.find_path(graph, start_row["node_id"], end_row["node_id"],
                                   departure_hour=8.0, v_free_bound_kmh=v_free_bound,
                                   corridor_bias=corridor_bias)

    # show_network=False: drawing the full ~18k-edge road network would add
    # ~13.6 MB of HTML per map to this saved notebook, and a worked example
    # only needs the route itself, its markers, and nearby POIs.
    demo_map = visualization.render_map(graph, pois, demo_result, departure_hour=8.0,
                                         show_network=False)
    display(demo_map)

    print(f"{start_name} -> {end_name}: {demo_result.total_distance_km:.2f} km, "
          f"{demo_result.total_travel_time_hours * 60:.1f} min")

Nhà hát Thành phố -> Bảo tàng Thành phố Hồ Chí Minh: 1.03 km, 3.5 min


Hồ Con Rùa -> Khách sạn Caravelle Saigon: 1.15 km, 3.5 min


Bệnh viện Từ Dũ -> Nhà thờ Thánh Jeanne d'Arc: 2.54 km, 6.0 min
